new version

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import os
import joblib
import warnings
warnings.filterwarnings("ignore")

# ========================= BEST CONFIG FROM YOUR RUN =========================
CHANNELS = list(range(41, 47))
CONTAMINATION = 0.05          # Best from your loop
DECISION_THRESHOLD = 0.0      # Best from your loop
RANDOM_SEED = 42

CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

print("=== Final Isolation Forest Baseline (Best Config) ===\n")

# Load cached train data
cache_file = os.path.join(CACHE_DIR, "preprocessed_train_channels41-46.pkl")
data_scaled, df_train, _ = joblib.load(cache_file)
print("Train data loaded from cache.")

# Load best model
model_file = os.path.join(CACHE_DIR, f"isolation_forest_cont{CONTAMINATION:.4f}.pkl")
model = joblib.load(model_file)
print(f"Model loaded (contamination={CONTAMINATION})")

# ========================= TEST INFERENCE =========================
print("Loading test.parquet...")
df_test = pd.read_parquet("/Users/alex/Downloads/esa-adb-challenge (1)/test.parquet")

channel_cols = [f'channel_{i}' for i in CHANNELS]
df_test = df_test[['id'] + channel_cols].copy()
df_test = df_test.sort_values('id').reset_index(drop=True)
df_test[channel_cols] = df_test[channel_cols].ffill().bfill()

test_data = df_test[channel_cols].values

print("Computing predictions on test...")
scores = model.decision_function(test_data)
anomaly_flag = (scores < DECISION_THRESHOLD).astype(int)

# ========================= SIMPLE POST-PROCESSING =========================
print("Applying simple post-processing (merge close detections)...")
# Convert to Series for easier rolling
anomaly_series = pd.Series(anomaly_flag, index=df_test['id'])

# Merge detections that are within 5 consecutive points
anomaly_series = anomaly_series.rolling(window=5, min_periods=1, center=True).max().astype(int)

# Optional: remove very short detections (less than 3 points)
anomaly_series = anomaly_series.rolling(window=3, min_periods=1, center=True).max().astype(int)

final_anomalies = anomaly_series.values

print(f"Predicted anomalies before post-processing: {anomaly_flag.sum()}")
print(f"Predicted anomalies after post-processing: {final_anomalies.sum()}")

# ========================= CREATE SUBMISSION =========================
submission = pd.DataFrame({
    'id': df_test['id'],
    'is_anomaly': final_anomalies
})

submission.to_parquet("submission_isoforest_best_pp_channels41-46.parquet", index=False)

print(f"\n✅ Submission saved: submission_isoforest_best_pp_channels41-46.parquet")
print(f"Final anomaly rate in submission: {final_anomalies.mean():.4%}")

print("\nUpload this file to Kaggle as Late Submission to get the real corrected event-wise F0.5 score.")

=== Final Isolation Forest Baseline (Best Config) ===

Train data loaded from cache.
Model loaded (contamination=0.05)
Loading test.parquet...
Computing predictions on test...
Applying simple post-processing (merge close detections)...
Predicted anomalies before post-processing: 16852
Predicted anomalies after post-processing: 79913

✅ Submission saved: submission_isoforest_best_pp_channels41-46.parquet
Final anomaly rate in submission: 15.3301%

Upload this file to Kaggle as Late Submission to get the real corrected event-wise F0.5 score.
